# Hydris - Aqueduct Engine Pilot
**Goal:** test every function the Aqueduct atlas performs, one at a time, in Colab.
Each function here becomes one FastAPI route later. Nothing heavy stored - all via Earth Engine.

Functions covered:
1. `get_site_profile(lat,lng)` - full 13-indicator risk profile for a point
2. `classify(score)` - score -> risk category (the scoring layer)
3. `reweight(profile, weights)` - custom weighting scheme (grouped + overall)
4. `get_future(pfaf_id, year, scenario)` - 2030/2050/2080 projections
5. `analyze_portfolio(sites)` - batch table for many sites
6. `search_location(name)` - geocode a place name -> basin
7. `indicator_layer(code)` + map - choropleth of all basins, clickable sites


## 0. Setup

In [ ]:
!pip install -q earthengine-api geemap geopy pandas
import ee, pandas as pd, pprint

EE_PROJECT = "your-gee-project-id"   # <-- EDIT
ee.Authenticate()
ee.Initialize(project=EE_PROJECT)

AQ     = ee.FeatureCollection("WRI/Aqueduct_Water_Risk/V4/baseline_annual")
AQ_FUT = ee.FeatureCollection("WRI/Aqueduct_Water_Risk/V4/future_annual")
print("ready")


## Config: the 13 indicators, their groups, and the category thresholds
Straight from the Aqueduct technical note.

In [ ]:
GROUPS = {
    "qan": ["bws","bwd","iav","sev","gtd","drr","rfr","cfr"],  # physical quantity
    "qal": ["ucw","cep"],                                       # physical quality
    "rrr": ["udw","usa","rri"],                                 # regulatory & reputational
}
ALL_IND = [c for g in GROUPS.values() for c in g]

CATEGORIES = [(0,1,"Low"),(1,2,"Low-Medium"),(2,3,"Medium-High"),(3,4,"High"),(4,5,"Extremely High")]

def clean(v):
    # -9999 = no data / insignificant -> None.  (9999 positive = arid mask, a real score 5, left alone)
    return None if v in (-9999, -9999.0, "-9999") else v

SITES = [
    {"id":"S1","name":"Chennai plant (IN)","lat":13.0827,"lng":80.2707},
    {"id":"S2","name":"Tiruppur textile (IN)","lat":11.1085,"lng":77.3411},
    {"id":"S3","name":"Fresno plant (US-CA)","lat":36.7378,"lng":-119.7871},
    {"id":"S4","name":"Hamburg plant (DE)","lat":53.5511,"lng":9.9937},
    {"id":"S5","name":"Riyadh plant (SA)","lat":24.7136,"lng":46.6753},
]
print(len(ALL_IND), "indicators across", len(GROUPS), "groups")


## Function 1 - `get_site_profile(lat, lng)`
Full risk profile: every indicator's raw / score / category / label, plus WRI's own overall.
(This is the atlas 'click a location' panel.)

In [ ]:
def get_site_profile(lat, lng, fc=AQ):
    pt  = ee.Geometry.Point([lng, lat])
    hit = fc.filterBounds(pt)
    if hit.size().getInfo() == 0:
        return None
    d = hit.first().toDictionary().getInfo()
    prof = {i: {"raw": clean(d.get(f"{i}_raw")), "score": clean(d.get(f"{i}_score")),
                "cat": d.get(f"{i}_cat"), "label": d.get(f"{i}_label")} for i in ALL_IND}
    prof["_meta"] = {"pfaf_id": d.get("pfaf_id"), "country": d.get("name_0"), "province": d.get("name_1")}
    prof["overall_gee"] = clean(d.get("w_awr_def_tot_score"))
    return prof

# TEST
prof = get_site_profile(13.0827, 80.2707)   # Chennai
print("basin:", prof["_meta"])
for i in ["bws","rfr","drr","gtd"]:
    print(f'  {i}: score={prof[i]["score"]}  ({prof[i]["label"]})')
print("WRI overall:", prof["overall_gee"])


## Function 2 - `classify(score)`
The scoring layer: turn a 0-5 score into its risk category. Validate against WRI's own labels.

In [ ]:
def classify(score):
    if score is None: return None
    for lo, hi, name in CATEGORIES:
        if lo <= score < hi: return name
    return "Extremely High"

# TEST: our classify() vs WRI's label for Chennai
for i in ["bws","rfr","drr"]:
    print(f'{i}: score={prof[i]["score"]:.2f} -> ours="{classify(prof[i]["score"])}" | WRI="{prof[i]["label"]}"')


## Function 3 - `reweight(profile, weights)`  *(the custom weighting scheme)*
Group score = weighted mean of its indicators. Overall = weighted mean of the 3 groups.
This is what powers the atlas 'custom weights' sliders. `None` scores are skipped and weights renormalized.

In [ ]:
def wmean(scores, weights):
    pairs = [(s, weights.get(k,0)) for k,s in scores.items() if s is not None and weights.get(k,0) > 0]
    if not pairs: return None
    num = sum(s*w for s,w in pairs); den = sum(w for _,w in pairs)
    return round(num/den, 3)

def reweight(profile, ind_weights=None, group_weights=None):
    ind_weights   = ind_weights   or {i: 1 for i in ALL_IND}          # base-2 scale: 0,.25,.5,1,2,4
    group_weights = group_weights or {"qan": 1, "qal": 1, "rrr": 1}
    scores = {i: profile[i]["score"] for i in ALL_IND}
    groups = {g: wmean({m: scores[m] for m in members}, ind_weights) for g, members in GROUPS.items()}
    overall = wmean(groups, group_weights)
    return {"groups": groups, "overall": overall, "overall_cat": classify(overall)}

# TEST: default (equal) weights vs a scarcity-heavy custom scheme
base = reweight(prof)
scarce = reweight(prof, ind_weights={**{i:1 for i in ALL_IND}, "bws":4, "bwd":4})
print("equal weights   -> overall", base["overall"], base["overall_cat"])
print("scarcity-heavy  -> overall", scarce["overall"], scarce["overall_cat"])
print("WRI default     -> overall", prof["overall_gee"], "(uses WRI's Delphi weights; plug those in to match exactly)")


## Function 4 - `get_future(pfaf_id, year, scenario)`
Projected scores for the 4 forward-modeled variables (stress, depletion, interannual/seasonal variability).
scenario: optimistic / bau / pessimistic ; year: 30 / 50 / 80.

In [ ]:
SCN = {"optimistic": "opt", "bau": "bau", "pessimistic": "pes"}
FUT = {"scarcity": "ws", "depletion": "wd", "interannual_var": "iv", "seasonal_var": "sv"}

def get_future(pfaf_id, year=50, scenario="bau", fc=AQ_FUT):
    f = fc.filter(ee.Filter.eq("pfaf_id", pfaf_id)).first()
    if f is None: return None
    keys = [f"{SCN[scenario]}{year}_{c}_x_s" for c in FUT.values()]
    d = f.toDictionary(keys).getInfo()
    return {name: clean(d.get(f"{SCN[scenario]}{year}_{c}_x_s")) for name, c in FUT.items()}

# TEST: Chennai now vs 2050 (business-as-usual)
pf = prof["_meta"]["pfaf_id"]
print("scarcity now :", prof["bws"]["score"])
print("2050 (bau)   :", get_future(pf, 50, "bau"))
print("2050 (pessim):", get_future(pf, 50, "pessimistic"))


## Function 5 - `analyze_portfolio(sites)`
Batch: many points -> one risk table. (The atlas 'analyze locations / upload' feature.)

In [ ]:
def analyze_portfolio(sites):
    rows = []
    for s in sites:
        p = get_site_profile(s["lat"], s["lng"])
        r = {"site": s["name"], "basin": p["_meta"]["pfaf_id"] if p else None}
        if p:
            r.update({"scarcity": p["bws"]["score"], "flood": p["rfr"]["score"],
                      "drought": p["drr"]["score"], "groundwater": p["gtd"]["score"],
                      "overall": p["overall_gee"]})
        rows.append(r)
    return pd.DataFrame(rows)

# TEST
analyze_portfolio(SITES).round(2)


## Function 6 - `search_location(name)`
Geocode a place name -> lat/lng -> basin. (The atlas search box.)

In [ ]:
from geopy.geocoders import Nominatim
_geo = Nominatim(user_agent="hydris-pilot")

def search_location(query):
    loc = _geo.geocode(query)
    if not loc: return None
    prof = get_site_profile(loc.latitude, loc.longitude)
    return {"query": query, "lat": loc.latitude, "lng": loc.longitude,
            "address": loc.address,
            "basin": prof["_meta"] if prof else None,
            "overall": prof["overall_gee"] if prof else None}

# TEST
pprint.pprint(search_location("Bengaluru, India"))


## Function 7 - `indicator_layer(code)` + interactive map
Choropleth of ALL basins for any indicator, with clickable site markers (data popup).

In [ ]:
import geemap

def indicator_layer(code="bws"):
    return AQ.reduceToImage([f"{code}_score"], ee.Reducer.first())

INDICATOR = "bws"   # try: bwd, rfr, drr, gtd, or w_awr_def_tot
palette = ["#2b83ba","#abdda4","#ffffbf","#fdae61","#d7191c"]

Map = geemap.Map(center=[20,78], zoom=4, basemap="CartoDB.Positron")
Map.addLayer(indicator_layer(INDICATOR), {"min":0,"max":5,"palette":palette}, f"{INDICATOR} (all basins)")
Map.addLayer(ee.Image().byte().paint(AQ, 1, 1), {"palette":["555555"]}, "basin outlines", True, 0.4)

rows = []
for s in SITES:
    p = get_site_profile(s["lat"], s["lng"]) or {}
    rows.append({"name": s["name"], "lat": s["lat"], "lng": s["lng"],
                 "scarcity": (p.get("bws") or {}).get("score"),
                 "overall": p.get("overall_gee")})
Map.add_points_from_xy(pd.DataFrame(rows), x="lng", y="lat", popup=["name","scarcity","overall"])
Map.add_colorbar({"min":0,"max":5,"palette":palette}, label=f"{INDICATOR} score")
Map


## Function 8 - `derive_pwi_scores(profile)` + `classify_risk_level(overall)`  *(Phase 0)*
Maps the WRI 13 indicators onto the **PWI framework dimensions** - Availability / Quality / Accessibility -
per the Water Stewardship Module spec, plus an auto risk-level badge.

Honesty notes baked in:
- `None` (no-data) scores stay `None` - they are **excluded and weights renormalized**, never treated as 0.
- The spec's 13th regulatory indicator `ety` (country regulatory) is not in the GEE Aqueduct table -> `rri` is used alone and `ety` is flagged as a roadmap item in `_notes`.

In [ ]:
def _wavg(pairs):
    """Weighted mean over (score, weight) pairs, skipping None scores and
    renormalizing weights over what's available. None if nothing available."""
    avail = [(s, w) for s, w in pairs if s is not None]
    if not avail: return None
    return round(sum(s*w for s, w in avail) / sum(w for _, w in avail), 2)

def _mx(*vals):
    vals = [v for v in vals if v is not None]
    return max(vals) if vals else None

def derive_pwi_scores(p):
    """WRI 13 indicators -> PWI framework dimensions (Availability / Quality / Accessibility).
    Weights per the Water Stewardship Module spec. No-data indicators are excluded
    with weight renormalization - never coerced to 0."""
    def s(k): return p[k]["score"]

    scarcity = _wavg([(s("bws"), .36), (s("bwd"), .36), (s("iav"), .18), (s("sev"), .10)])

    availability = {
        "physical_scarcity": scarcity,
        "flood":             _mx(s("rfr"), s("cfr")),      # spec: MAX(rfr, cfr)
        "drought":           s("drr"),
        "groundwater":       s("gtd"),
    }
    quality = {
        "untreated_wastewater": s("ucw"),
        "coastal_eutrophication": s("cep"),
        "reg_reputational":     s("rri"),                  # spec wants MAX(rri, ety); ety not in GEE
    }
    accessibility = {
        "wash_gap": _wavg([(s("udw"), .5), (s("usa"), .5)]),  # spec: AVG(udw, usa)
    }
    return {
        "Availability":  availability,
        "Quality":       quality,
        "Accessibility": accessibility,
        # dimension rollups (equal-weight mean of available components) for the ring gauges
        "dims": {
            "Availability":  _wavg([(v, 1) for v in availability.values()]),
            "Quality":       _wavg([(v, 1) for v in quality.values()]),
            "Accessibility": _wavg([(v, 1) for v in accessibility.values()]),
        },
        "derived": {
            "flood":    _mx(s("rfr"), s("cfr")),
            "shortage": _wavg([(s("bws"), .5), (s("drr"), .5)]),
        },
        "_notes": [
            "ety (country regulatory) not available in GEE Aqueduct table - rri used alone; ety flagged as roadmap",
            "no-data indicators excluded with weight renormalization, never treated as 0",
        ],
    }

def classify_risk_level(overall):
    """0-5 score -> WRI risk-level badge. None-safe."""
    if overall is None: return None
    return ("Extremely High" if overall >= 4 else "High" if overall >= 3
            else "Medium-High" if overall >= 2 else "Low-Medium" if overall >= 1 else "Low")

print("Phase-0 functions defined")

In [ ]:
# ✅ GATE 0 - run on all 5 sites. PASS requires:
#   1. all 3 PWI dims populated (or explicitly None with a reason, e.g. Riyadh)
#   2. badge == WRI's own overall category (w_awr_def_tot_cat direction)
#   3. ZERO -9999 anywhere; Chennai gtd / Riyadh drr must print 'no data', never 0
import json

def _fmt(v): return "no data" if v is None else v

gate0_pass = True
for st in SITES:
    p = get_site_profile(st["lat"], st["lng"])
    if not p:
        print(st["name"], "-> NO BASIN"); continue
    pwi = derive_pwi_scores(p)
    lvl = classify_risk_level(p["overall_gee"])

    # check 3: -9999 leak scan over the whole output
    leak = "-9999" in json.dumps(pwi)
    if leak: gate0_pass = False

    print(f'\n{st["name"]}  overall={_fmt(p["overall_gee"])}  badge={lvl}  (WRI cat sanity: {p["bws"]["label"]}...)')
    print("  dims:", {k: _fmt(v) for k, v in pwi["dims"].items()})
    print("  Availability :", {k: _fmt(v) for k, v in pwi["Availability"].items()})
    print("  Quality      :", {k: _fmt(v) for k, v in pwi["Quality"].items()})
    print("  Accessibility:", {k: _fmt(v) for k, v in pwi["Accessibility"].items()})
    if leak: print("  ❌ -9999 LEAK DETECTED")

# spot-assert the two known no-data cases
chennai = derive_pwi_scores(get_site_profile(13.0827, 80.2707))
riyadh  = derive_pwi_scores(get_site_profile(24.7136, 46.6753))
print("\nChennai groundwater (expect None/no-data):", _fmt(chennai["Availability"]["groundwater"]))
print("Riyadh drought       (expect None/no-data):", _fmt(riyadh["Availability"]["drought"]))
print("\nGATE 0:", "✅ PASS (verify dims + badges above by eye)" if gate0_pass else "❌ FAIL (-9999 leak)")

---
**Each function above = one API route next.** Once they all pass here:
- `get_site_profile` -> `GET /site?lat=&lng=`
- `reweight` -> `POST /reweight`
- `get_future` -> `GET /future?pfaf_id=&year=&scenario=`
- `analyze_portfolio` -> `POST /portfolio`
- `search_location` -> `GET /search?q=`
- `indicator_layer` -> `GET /tiles/{indicator}`

**Data:** WRI Aqueduct 4.0 via Google Earth Engine, free with attribution to WRI.